# Import Zone

In [ ]:
from qutip import *
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib
import sys
from pathlib import Path
import json
from lmfit import Model
from matplotlib.ticker import MaxNLocator


import scienceplots as splt
matplotlib.pyplot.style.use(['science', 'notebook', 'ieee'])
matplotlib.pyplot.rcParams['font.family'] = 'Times New Roman'


project_root = Path.cwd().parent.parent.parent 
sys.path.append(str(project_root / "src"))


from molecules.molecule import CaH, CaOH, CaOH_dm2, CaH_dm2

from pumping.pump_evolution import run_pumping
from state_preparation import run_pumping_pipeline
from BayesianSE.main import run_bayesian_state_estimation
from BayesianSE._plotting import plot_method_comparison

# Molecule creation 

In [ ]:
b_field_gauss = 3.6
j_max = 14

molecule = CaH.create_molecule_data(b_field_gauss=b_field_gauss, j_max=j_max)
molecule_type = "CaH"

temperature = 300

plot = True

# Pumping and RAP stages

In [ ]:
results = run_pumping_pipeline(
    molecule_type="CaH",
    molecule=molecule,
    temperature=temperature,
    j_max=j_max,
    plot=plot
)

pop0_before, pop1_before, pop2jp1_before = results["before"]

pop0_after, pop1_after, pop2jp1_after = results["after"]

pop_fit_0, pop_fit_1, pop_fit_2jp1 = results["pop_fit"]

pop_fit = [pop_fit_0, pop_fit_1, pop_fit_2jp1]

# Bayesian state estimation


In [ ]:
apply_pumping = False
marginalization = False
false_rates = True


false_positive_rate = 0.03
false_negative_rate = 0.03


rabi_by_j = 2 * np.pi * 0.002


dephased = False
coherence_time_us = 1000
is_minus = False

# LASER MISCALIBRATION
laser_miscalibration = {
    "frequency": {"type": "abs_gaussian", "level": 0.00005},
    "rabi_rate": {"type": "rel_gaussian", "level": 0.1}
}


# SHOT TO SHOT FLUCTUATIONS
noise_params = {
    "frequency": {"type": "abs_gaussian", "level": 0.00003},
    "rabi_rate": {"type": "rel_gaussian", "level": 0.01}
}


# LASER MISCALIBRATION
laser_miscalibration_estim = {
    "frequency": {"type": "abs_gaussian", "level": 0.00000},
    "rabi_rate": {"type": "rel_gaussian", "level": 0.0}
}


# SHOT TO SHOT FLUCTUATIONS
noise_params_estim = {
    "frequency": {"type": "rel_gaussian", "level": 0.000},
    "rabi_rate": {"type": "rel_gaussian", "level": 0.000}
}

### Block 0 (single measurement shots)

In [ ]:
N = 1000
num_updates = 300
block_steps = 1
type_block = None

results=run_bayesian_state_estimation(molecule=molecule, molecule_type=molecule_type, temperature=temperature, b_field_gauss=b_field_gauss, j_max=j_max,
                              rabi_by_j=rabi_by_j, dephased=dephased, coherence_time_us=coherence_time_us, is_minus=is_minus,
                              false_positive_rate=false_positive_rate, false_negative_rate=false_negative_rate,
                              noise_params=noise_params, seed=None, laser_miscalibration=laser_miscalibration, seed_miscalibration=None,
                              noise_params_estim=noise_params_estim, laser_miscalibration_estim=laser_miscalibration_estim,
                              pop_fit=pop_fit,
                              N=N, num_updates=num_updates, block_steps=block_steps, type_block=type_block, 
                              apply_pumping=apply_pumping, marginalization=marginalization, false_rates=false_rates,
                              save_data=True, only_total=False
                              )

stats_bl0 = results["stats"]
success_rate_total_bl0 = results["success_rate_total"]
stats_filtered_bl0 = results["stats_filtered"]
success_rate_filtered_bl0 = results["success_rate_filtered"]

### Block 1

In [ ]:
N = 1000
num_updates = 200
block_steps = 3
type_block = "block1"

results=run_bayesian_state_estimation(molecule=molecule, molecule_type=molecule_type, temperature=temperature, b_field_gauss=b_field_gauss, j_max=j_max,
                              rabi_by_j=rabi_by_j, dephased=dephased, coherence_time_us=coherence_time_us, is_minus=is_minus,
                              false_positive_rate=false_positive_rate, false_negative_rate=false_negative_rate,
                              noise_params=noise_params, seed=None, laser_miscalibration=laser_miscalibration, seed_miscalibration=None,
                              noise_params_estim=noise_params_estim, laser_miscalibration_estim=laser_miscalibration_estim,
                              pop_fit=pop_fit,
                              N=N, num_updates=num_updates, block_steps=block_steps, type_block=type_block, 
                              apply_pumping=apply_pumping, marginalization=marginalization, false_rates=false_rates,
                              save_data=True, only_total=False
                              )

stats_bl1 = results["stats"]
success_rate_total_bl1 = results["success_rate_total"]
stats_filtered_bl1 = results["stats_filtered"]
success_rate_filtered_bl1 = results["success_rate_filtered"]

### Block 2

In [ ]:
N = 1000
num_updates = 200
block_steps = 3
type_block = "block2"

results=run_bayesian_state_estimation(molecule=molecule, molecule_type=molecule_type, temperature=temperature, b_field_gauss=b_field_gauss, j_max=j_max,
                              rabi_by_j=rabi_by_j, dephased=dephased, coherence_time_us=coherence_time_us, is_minus=is_minus,
                              false_positive_rate=false_positive_rate, false_negative_rate=false_negative_rate,
                              noise_params=noise_params, seed=None, laser_miscalibration=laser_miscalibration, seed_miscalibration=None,
                              noise_params_estim=noise_params_estim, laser_miscalibration_estim=laser_miscalibration_estim,
                              pop_fit=pop_fit,
                              N=N, num_updates=num_updates, block_steps=block_steps, type_block=type_block, 
                              apply_pumping=apply_pumping, marginalization=marginalization, false_rates=false_rates,
                              save_data=True, only_total=False
                              )

stats_bl2 = results["stats"]
success_rate_total_bl2 = results["success_rate_total"]
stats_filtered_bl2 = results["stats_filtered"]
success_rate_filtered_bl2 = results["success_rate_filtered"]

### Block 3

In [ ]:
N = 1000
num_updates = 300
block_steps = 2
type_block = "block3"

results=run_bayesian_state_estimation(molecule=molecule, molecule_type=molecule_type, temperature=temperature, b_field_gauss=b_field_gauss, j_max=j_max,
                              rabi_by_j=rabi_by_j, dephased=dephased, coherence_time_us=coherence_time_us, is_minus=is_minus,
                              false_positive_rate=false_positive_rate, false_negative_rate=false_negative_rate,
                              noise_params=noise_params, seed=None, laser_miscalibration=laser_miscalibration, seed_miscalibration=None,
                              noise_params_estim=noise_params_estim, laser_miscalibration_estim=laser_miscalibration_estim,
                              pop_fit=pop_fit,
                              N=N, num_updates=num_updates, block_steps=block_steps, type_block=type_block, 
                              apply_pumping=apply_pumping, marginalization=marginalization, false_rates=false_rates,
                              save_data=True, only_total=False
                              )

stats_bl3 = results["stats"]
success_rate_total_bl3 = results["success_rate_total"]
stats_filtered_bl3 = results["stats_filtered"]
success_rate_filtered_bl3 = results["success_rate_filtered"]

### Block 4

In [ ]:
N = 1000
num_updates = 150
block_steps = 4
type_block = "block4"

results=run_bayesian_state_estimation(molecule=molecule, molecule_type=molecule_type, temperature=temperature, b_field_gauss=b_field_gauss, j_max=j_max,
                              rabi_by_j=rabi_by_j, dephased=dephased, coherence_time_us=coherence_time_us, is_minus=is_minus,
                              false_positive_rate=false_positive_rate, false_negative_rate=false_negative_rate,
                              noise_params=noise_params, seed=None, laser_miscalibration=laser_miscalibration, seed_miscalibration=None,
                              noise_params_estim=noise_params_estim, laser_miscalibration_estim=laser_miscalibration_estim,
                              pop_fit=pop_fit,
                              N=N, num_updates=num_updates, block_steps=block_steps, type_block=type_block, 
                              apply_pumping=apply_pumping, marginalization=marginalization, false_rates=false_rates,
                              save_data=True, only_total=False
                              )

stats_bl4 = results["stats"]
success_rate_total_bl4 = results["success_rate_total"]
stats_filtered_bl4 = results["stats_filtered"]
success_rate_filtered_bl4 = results["success_rate_filtered"]

### Block 5

In [ ]:
N = 1000
num_updates = 100
block_steps = 6
type_block = "block5"

results=run_bayesian_state_estimation(molecule=molecule, molecule_type=molecule_type, temperature=temperature, b_field_gauss=b_field_gauss, j_max=j_max,
                              rabi_by_j=rabi_by_j, dephased=dephased, coherence_time_us=coherence_time_us, is_minus=is_minus,
                              false_positive_rate=false_positive_rate, false_negative_rate=false_negative_rate,
                              noise_params=noise_params, seed=None, laser_miscalibration=laser_miscalibration, seed_miscalibration=None,
                              noise_params_estim=noise_params_estim, laser_miscalibration_estim=laser_miscalibration_estim,
                              pop_fit=pop_fit,
                              N=N, num_updates=num_updates, block_steps=block_steps, type_block=type_block, 
                              apply_pumping=apply_pumping, marginalization=marginalization, false_rates=false_rates,
                              save_data=True, only_total=False
                              )

stats_bl5 = results["stats"]
success_rate_total_bl5 = results["success_rate_total"]
stats_filtered_bl5 = results["stats_filtered"]
success_rate_filtered_bl5 = results["success_rate_filtered"]

### Block 6

In [ ]:
N = 1000
num_updates = 75
block_steps = 9
type_block = "block6"

results=run_bayesian_state_estimation(molecule=molecule, molecule_type=molecule_type, temperature=temperature, b_field_gauss=b_field_gauss, j_max=j_max,
                              rabi_by_j=rabi_by_j, dephased=dephased, coherence_time_us=coherence_time_us, is_minus=is_minus,
                              false_positive_rate=false_positive_rate, false_negative_rate=false_negative_rate,
                              noise_params=noise_params, seed=None, laser_miscalibration=laser_miscalibration, seed_miscalibration=None,
                              noise_params_estim=noise_params_estim, laser_miscalibration_estim=laser_miscalibration_estim,
                              pop_fit=pop_fit,
                              N=N, num_updates=num_updates, block_steps=block_steps, type_block=type_block, 
                              apply_pumping=apply_pumping, marginalization=marginalization, false_rates=false_rates,
                              save_data=True, only_total=False
                              )

stats_bl6 = results["stats"]
success_rate_total_bl6 = results["success_rate_total"]
stats_filtered_bl6 = results["stats_filtered"]
success_rate_filtered_bl6 = results["success_rate_filtered"]

### Block 7

In [ ]:
N = 1000
num_updates = 50
block_steps = 12
type_block = "block7"

results=run_bayesian_state_estimation(molecule=molecule, molecule_type=molecule_type, temperature=temperature, b_field_gauss=b_field_gauss, j_max=j_max,
                              rabi_by_j=rabi_by_j, dephased=dephased, coherence_time_us=coherence_time_us, is_minus=is_minus,
                              false_positive_rate=false_positive_rate, false_negative_rate=false_negative_rate,
                              noise_params=noise_params, seed=None, laser_miscalibration=laser_miscalibration, seed_miscalibration=None,
                              noise_params_estim=noise_params_estim, laser_miscalibration_estim=laser_miscalibration_estim,
                              pop_fit=pop_fit,
                              N=N, num_updates=num_updates, block_steps=block_steps, type_block=type_block, 
                              apply_pumping=apply_pumping, marginalization=marginalization, false_rates=false_rates,
                              save_data=True, only_total=False
                              )

stats_bl7 = results["stats"]
success_rate_total_bl7 = results["success_rate_total"]
stats_filtered_bl7 = results["stats_filtered"]
success_rate_filtered_bl7 = results["success_rate_filtered"]

### Block 8

In [ ]:
N = 1000
num_updates = 25
block_steps = 24
type_block = "block8"

results=run_bayesian_state_estimation(molecule=molecule, molecule_type=molecule_type, temperature=temperature, b_field_gauss=b_field_gauss, j_max=j_max,
                              rabi_by_j=rabi_by_j, dephased=dephased, coherence_time_us=coherence_time_us, is_minus=is_minus,
                              false_positive_rate=false_positive_rate, false_negative_rate=false_negative_rate,
                              noise_params=noise_params, seed=None, laser_miscalibration=laser_miscalibration, seed_miscalibration=None,
                              noise_params_estim=noise_params_estim, laser_miscalibration_estim=laser_miscalibration_estim,
                              pop_fit=pop_fit,
                              N=N, num_updates=num_updates, block_steps=block_steps, type_block=type_block, 
                              apply_pumping=apply_pumping, marginalization=marginalization, false_rates=false_rates,
                              save_data=True, only_total=False
                              )

stats_bl8 = results["stats"]
success_rate_total_bl8 = results["success_rate_total"]
stats_filtered_bl8 = results["stats_filtered"]
success_rate_filtered_bl8 = results["success_rate_filtered"]

### Block 9

In [ ]:
N = 1000
num_updates = 25
block_steps = 24
type_block = "block9"

results=run_bayesian_state_estimation(molecule=molecule, molecule_type=molecule_type, temperature=temperature, b_field_gauss=b_field_gauss, j_max=j_max,
                              rabi_by_j=rabi_by_j, dephased=dephased, coherence_time_us=coherence_time_us, is_minus=is_minus,
                              false_positive_rate=false_positive_rate, false_negative_rate=false_negative_rate,
                              noise_params=noise_params, seed=None, laser_miscalibration=laser_miscalibration, seed_miscalibration=None,
                              noise_params_estim=noise_params_estim, laser_miscalibration_estim=laser_miscalibration_estim,
                              pop_fit=pop_fit,
                              N=N, num_updates=num_updates, block_steps=block_steps, type_block=type_block, 
                              apply_pumping=apply_pumping, marginalization=marginalization, false_rates=false_rates,
                              save_data=True, only_total=False
                              )

stats_bl9 = results["stats"]
success_rate_total_bl9 = results["success_rate_total"]
stats_filtered_bl9 = results["stats_filtered"]
success_rate_filtered_bl9 = results["success_rate_filtered"]

# Comparison

In [ ]:
stats_list = [stats_bl0, stats_bl1, stats_bl2, stats_bl3, stats_bl4, stats_bl5, stats_bl6, stats_bl7, stats_bl8, stats_bl9]
method_names = ["Block 0", "Block 1", "Block 2", "Block 3", "Block 4", "Block 5", "Block 6", "Block 7", "Block 8", "Block 9"]

success_rate_total = [success_rate_total_bl0, success_rate_total_bl1, success_rate_total_bl2, success_rate_total_bl3, success_rate_total_bl4, 
                      success_rate_total_bl5, success_rate_total_bl6, success_rate_total_bl7, success_rate_total_bl8, success_rate_total_bl9]


plot_method_comparison(stats_list, method_names, success_rate_total, N, filename="method_comparison_2_cah.svg")